# 10 - השוואה למודלים של רשתות אקראיות

מספר בודד כגון "average clustering = 0.02" אינו אומר דבר בפני עצמו. כדי לקבוע האם רשת תחנות התחבורה הציבורית בישראל היא *חריגה* מבחינת clustering, *חריגה* מבחינת הטרוגניות או *חריגה* במרחקה ממבנה small world, נדרשים מודלי אפס (null models): גרפים סינתטיים בעלי **אותו גודל**, הנוצרים על ידי מנגנון ידוע. מחברת זו בונה מחדש (או טוענת) את גרף התחנות הלא-מכוון האמיתי ומשווה אותו לארבעה מודלים קלאסיים - Erdos-Renyi `G(n, m)`, configuration model על רצף הדרגות האמיתי, Barabasi-Albert של preferential attachment, ו-Watts-Strogatz של small world - במונחי התפלגות דרגות, clustering ואורך מסלול קצר ביותר ממוצע.

**שאלת המחקר:** איזה מנגנון גנרטיבי, אם בכלל, משחזר את מבנה רשת התחנות האמיתית של אוטובוסים ורכבות - ומה מלמד אותנו חוסר ההתאמה על הגורמים שמעצבים בפועל את הרשת?

**קלט**
- `outputs/nb/02_graph_construction/**` - גרף התחנות שנשמר על ידי המחברת `02_graph_construction` (בפורמט GraphML או כקובץ CSV של רשימת קשתות), אם הוא קיים.
- `israel-public-transportation/stop_times.txt` - מסלול גיבוי בלבד. אם שלב 02 לא הורץ, המחברת בונה מחדש את הגרף ישירות מתוך פיד ה-GTFS הגולמי (816 MB, מורד לפי הצורך).

**פלט** (כל הקבצים תחת `outputs/nb/10_network_model_comparison/`)
- `tables/network_model_comparison.csv` - שורה אחת לכל גרף, הכוללת את מספרי הצמתים והקשתות שהושגו בפועל ואת כל מדדי המבנה.
- `tables/degree_distributions.csv` - התפלגות הדרגות המלאה (P(k) ו-CCDF) עבור כל גרף.
- `tables/degree_distribution_summary.csv` - מומנטים של הדרגות, אומדן למעריך הזנב ומרחק KS מהתפלגות הדרגות האמיתית.
- `tables/small_world_ratios.csv` - C/C_random, L/L_random ומדד ה-small-world סיגמא.
- `figures/degree_distribution_comparison.png`, `figures/model_metric_comparison.png`, `figures/size_match_check.png`.

**באג שמחברת זו מתקנת.** הסקריפט המקורי בחר את הפרמטר של Barabasi-Albert כ-`m = round(edges / nodes) = 2`, בחירה המייצרת כ-61k קשתות מול 51.8k האמיתיות, וכפה על הפרמטר `k` של Watts-Strogatz להיות זוגי, מה שהוביל לקריסתו ל-`k = 2`, כלומר טבעת חשופה בעלת כ-30.5k קשתות. השוואת clustering ואורך מסלול בין גרפים שמספרי הקשתות שלהם נבדלים ב-20-40% היא חסרת משמעות: שני הגדלים תלויים ישירות בצפיפות הקשתות. בהמשך, שני הגנרטורים מוחלפים בגרסאות מותאמות-גודל הפוגעות במדויק במספר הקשתות האמיתי, וכל מודל מדווח את מספרי הצמתים והקשתות שהושגו בפועל, כך שכל אי-התאמה שנותרה (ה-configuration model מאבד בהכרח מספר קשתות) נותרת גלויה.

## 1. אתחול סביבת העבודה

תא זה מאפשר להריץ את המחברת הן על עותק מקומי של המאגר והן ב-Google Colab. הוא מתקין רק את החבילות החסרות בפועל, מאתר את שורש המאגר על ידי חיפוש תיקיית ה-GTFS בשם `israel-public-transportation` (ומשכפל את המאגר אם אנו פועלים ב-Colab), ויוצר את התיקייה המשותפת `outputs/nb` שאליה כותבות כל המחברות בסדרה זו.

In [ ]:
# --- Environment bootstrap (safe to re-run, works locally and on Google Colab) ---
import os, sys, subprocess
from pathlib import Path

def _ensure(*pkgs):
    """Install only the packages that are actually missing."""
    import importlib.util
    alias = {"scikit-learn": "sklearn", "python-louvain": "community",
             "python-bidi": "bidi", "node2vec": "node2vec"}
    missing = [p for p in pkgs
               if importlib.util.find_spec(alias.get(p, p.replace("-", "_"))) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

def find_repo_root():
    """Find the repo locally; on Colab, clone it."""
    here = Path(os.getcwd()).resolve()
    for cand in [here, *here.parents]:
        if (cand / "israel-public-transportation").is_dir():
            return cand
    target = Path("/content/israel-transit-network-resilience")
    if not target.exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/seanfourman/israel-transit-network-resilience.git",
                        str(target)], check=True)
    return target

REPO = find_repo_root()
os.chdir(REPO)
DATA = REPO / "israel-public-transportation"
OUT = REPO / "outputs" / "nb"
OUT.mkdir(parents=True, exist_ok=True)
print("Repo root:", REPO)

## 2. ייבוא ספריות, קבועי כוונון ותיקיות השלב

כל הפרמטרים היקרים מרוכזים כאן, כך שניתן לכוונן את זמן הריצה ללא נגיעה בקוד הניתוח:

- `PATH_SOURCE_SAMPLES = 128` - חישוב מדויק של מסלולים קצרים ביותר בין כל הזוגות על כ-30k צמתים אינו אפשרי (כ-4.6e8 זוגות). אנו מריצים BFS מ-128 מקורות אקראיים לכל גרף וממצעים את המרחקים שהושגו. העלות היא כ-128 סריקות BFS x 5 גרפים, מספר שניות לכל אחת; העלאת הערך ל-512 משפרת את הדיוק במחיר של פי 4 בקירוב.
- `CLUSTERING_TRIALS = 2000` - משמש רק כמסלול גיבוי אם חישוב clustering מדויק נחשב יקר מדי (ראו `CLUSTERING_EXACT_BUDGET`).
- `CLUSTERING_EXACT_BUDGET` - קירוב לעלות החישוב (`sum of degree^2`). מתחת לערך זה אנו מחשבים clustering באופן מדויק, חישוב שהוא זול בדילול הנוכחי ומסלק רעש דגימה מההשוואה המרכזית.
- `WS_REWIRE_P = 0.05` - הסתברות ה-rewiring של Watts-Strogatz ששימשה בסקריפט המקורי; נשמרה לשם רציפות.
- `SEED = 42` - כל גנרטור וכל דוגם מאותחלים בזרע קבוע, כך שהמחברת כולה ניתנת לשחזור.

בהתאם למוסכמת הפלט של הפרויקט, מחברת זו כותבת רק לתיקיית השלב שלה ואינה נוגעת ב-`outputs/tables`, `outputs/figures` או `outputs/rail`.

In [ ]:
_ensure('networkx', 'pandas', 'matplotlib', 'numpy')

import csv
import math
import random
from collections import Counter

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd

SEED = 42
PATH_SOURCE_SAMPLES = 128        # BFS sources per graph for path-length statistics
CLUSTERING_TRIALS = 2000         # fallback sample size for approximate clustering
CLUSTERING_EXACT_BUDGET = 5_000_000  # sum(degree^2) below which exact clustering is affordable
WS_REWIRE_P = 0.05               # Watts-Strogatz rewiring probability
K_MIN_TAIL = 4                   # lower cut-off for the descriptive tail-exponent estimate

STAGE = OUT / '10_network_model_comparison'
TABLES = STAGE / 'tables'
FIGURES = STAGE / 'figures'
TABLES.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

PREV_STAGE = OUT / '02_graph_construction'

pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 40)
print('Stage output folder:', STAGE)

## 3. מהיכן מגיע הגרף האמיתי?

מחברת זו היא צרכנית של גרף התחנות שנבנה במחברת `02_graph_construction`. כדי לשמור על עמידות, אנו בודקים תחילה בתיקיית אותו שלב אם קיים בה ארטיפקט גרף כלשהו (קובץ `.graphml`/`.gexf`, או קובץ CSV ששמו מכיל את המחרוזת `edge`). אם שלב 02 לא הורץ, אנו נופלים חזרה לבנייה מחדש של הגרף מפיד ה-GTFS הגולמי לפי בדיוק אותו כלל בנייה, כך שבודק יוכל להריץ מחברת זו באופן עצמאי. התא שלהלן רק *מחליט* באיזה מסלול לבחור ומדפיס את ההחלטה.

In [ ]:
def find_stage02_graph():
    '''Return a graph artifact saved by notebook 02, or None if stage 02 is absent.'''
    if not PREV_STAGE.exists():
        return None
    for pattern in ('**/*.graphml', '**/*.gexf'):
        for path in sorted(PREV_STAGE.glob(pattern)):
            return path
    for path in sorted(PREV_STAGE.glob('**/*.csv')):
        if 'edge' in path.name.lower():
            return path
    return None


GRAPH_SOURCE = find_stage02_graph()
if GRAPH_SOURCE is None:
    print('No artifact found in', PREV_STAGE)
    print('Falling back to rebuilding the stop graph from the raw GTFS feed.')
    print('(Run notebook 02_graph_construction first if you prefer to reuse its output.)')
else:
    print('Reusing stage 02 artifact:', GRAPH_SOURCE)

## 4. הפיד הגולמי (מסלול הגיבוי בלבד)

הקובץ `stop_times.txt` שוקל 816 MB ואינו נכלל במעקב git במכוון. הוא נחוץ רק כאשר עלינו לבנות מחדש את הגרף בעצמנו, ולכן ההורדה מדולגת לחלוטין כאשר נמצא ארטיפקט של שלב 02. ההורדה נמשכת מספר דקות בהרצה הראשונה ונשמרת במטמון על הדיסק לאחר מכן.

In [ ]:
STOP_TIMES = DATA / 'stop_times.txt'

if GRAPH_SOURCE is None:
    # stop_times.txt is 816MB and is not tracked in git - fetch it on demand.
    _ensure('gdown')
    import gdown
    if not STOP_TIMES.exists():
        gdown.download(id='1V_yPAWXV6mGTFGrfiosah5LngcLZnviW',
                       output=str(STOP_TIMES), quiet=False)
    print('stop_times.txt:', round(STOP_TIMES.stat().st_size / 1024**2, 1), 'MB')
else:
    print('Raw feed not needed - the graph comes from stage 02.')

## 5. בניית (או טעינת) גרף התחנות הלא-מכוון האמיתי

כלל המידול, המועתק מילה במילה מסקריפט הניתוח של הפרויקט: **הצמתים הם תחנות GTFS, וקשת מחברת שתי תחנות העוקבות זו לזו בתוך אותו trip**. הפיד ממוין לפי `trip_id` ולאחר מכן לפי `stop_sequence`, ולכן די במעבר יחיד בזרימה (streaming) על פני 15.7M השורות - אנו לעולם לא מחזיקים את הקובץ כולו בזיכרון. מקטעים מקבילים מקוצים יחד וספירותיהם מסוכמות לתוך תכונת הקשת `frequency` (כמה מקטעי trip מתוזמנים עושים שימוש באותו זוג תחנות). הכיווניות מושמטת כאן משום ש-clustering, אורך מסלול והתפלגות דרגות מוגדרים כולם על הגרף הלא-מכוון.

צפויים כ-30k צמתים וכ-52k קשתות. הבנייה מחדש בזרימה נמשכת כ-2-4 דקות; טעינת ארטיפקט של שלב 02 היא כמעט מיידית.

In [ ]:
def gtfs_rows(path):
    '''Stream a GTFS csv file row by row (never loads the whole file).'''
    with path.open('r', encoding='utf-8-sig', newline='') as handle:
        yield from csv.DictReader(handle)


def build_undirected_stop_graph(stop_times_path):
    '''Consecutive stops within a trip become an undirected weighted edge.'''
    edge_counts = Counter()
    stop_use_counts = Counter()
    previous_trip_id = None
    previous_stop_id = None
    rows_seen = 0

    for row in gtfs_rows(stop_times_path):
        rows_seen += 1
        trip_id = row['trip_id']
        stop_id = row['stop_id']
        stop_use_counts[stop_id] += 1
        if (previous_trip_id == trip_id
                and previous_stop_id is not None
                and previous_stop_id != stop_id):
            edge_counts[(previous_stop_id, stop_id)] += 1
        previous_trip_id = trip_id
        previous_stop_id = stop_id

    graph = nx.Graph()
    graph.add_nodes_from(stop_use_counts)
    for (source, target), frequency in edge_counts.items():
        if graph.has_edge(source, target):
            graph[source][target]['frequency'] += frequency
        else:
            graph.add_edge(source, target, frequency=frequency)
    print(f'stop_times rows streamed: {rows_seen:,}')
    return graph


def load_edge_list_csv(path):
    '''Load an undirected graph from a stage-02 edge-list csv with flexible column names.'''
    frame = pd.read_csv(path, dtype=str, keep_default_na=False)
    columns = {name.lower(): name for name in frame.columns}
    source_col = next((columns[c] for c in
                       ('source', 'source_stop_id', 'from_stop_id', 'from_stop', 'stop_a', 'u')
                       if c in columns), None)
    target_col = next((columns[c] for c in
                       ('target', 'target_stop_id', 'to_stop_id', 'to_stop', 'stop_b', 'v')
                       if c in columns), None)
    if source_col is None or target_col is None:
        raise ValueError(
            f'{path} does not look like an edge list (columns: {list(frame.columns)}). '
            'Re-run notebook 02_graph_construction, or delete the file so this notebook '
            'rebuilds the graph from stop_times.txt.')
    weight_col = next((columns[c] for c in ('trip_frequency', 'frequency', 'weight', 'trips', 'count')
                       if c in columns), None)
    graph = nx.Graph()
    for _, row in frame.iterrows():
        source, target = row[source_col], row[target_col]
        weight = int(float(row[weight_col])) if weight_col else 1
        if graph.has_edge(source, target):
            graph[source][target]['frequency'] += weight
        else:
            graph.add_edge(source, target, frequency=weight)
    return graph


if GRAPH_SOURCE is None:
    G = build_undirected_stop_graph(STOP_TIMES)
elif GRAPH_SOURCE.suffix == '.graphml':
    G = nx.Graph(nx.read_graphml(GRAPH_SOURCE))
elif GRAPH_SOURCE.suffix == '.gexf':
    G = nx.Graph(nx.read_gexf(GRAPH_SOURCE))
else:
    G = load_edge_list_csv(GRAPH_SOURCE)

G.remove_edges_from(nx.selfloop_edges(G))
N_REAL = G.number_of_nodes()
M_REAL = G.number_of_edges()
if N_REAL == 0 or M_REAL == 0:
    raise RuntimeError('The loaded stop graph is empty - check the stage 02 artifact or the GTFS feed.')
print(f'Real transit graph: {N_REAL:,} nodes, {M_REAL:,} undirected edges, '
      f'average degree {2 * M_REAL / N_REAL:.3f}')

## 6. פונקציות עזר למדידה (טיפול זהה עבור כל גרף)

כדי שההשוואה תהיה הוגנת, כל גרף - אמיתי או סינתטי - חייב להימדד בדיוק על ידי אותו קוד, אותו גודל מדגם ואותו זרע. פונקציות העזר הבאות עושות זאת:

- `sampled_path_statistics` מריצה BFS ממספר קבוע של מקורות אקראיים **בתוך הרכיב הקשיר הגדול ביותר** (זוג בלתי-נגיש הוא בעל מרחק אינסופי, מה שהיה הופך את הממוצע ללא מוגדר) וממצעת את המרחקים שהושגו. המרחק המקסימלי שנצפה מדווח כאומדן לקוטר - הוא חסם תחתון לקוטר האמיתי.
- `average_clustering_value` מעדיפה את מקדם ה-clustering המקומי הממוצע המדויק, ונופלת חזרה לאומד הדגימה של networkx רק כאשר הגרף צפוף די הצורך כדי להאט את החישוב המדויק. השיטה שבה נעשה שימוש נרשמת בטבלת התוצאות כך ששום דבר אינו מוסתר.
- `graph_property_row` מייצרת שורה אחת של טבלת ההשוואה, ומתחילה במכוון ב-`nodes`, ב-`edges` וביחסים מול הגרף האמיתי, כך שכל אי-התאמה בגודל היא הדבר הראשון שהקורא רואה.

In [ ]:
def largest_component_subgraph(graph):
    '''Copy of the largest connected component.'''
    largest_nodes = max(nx.connected_components(graph), key=len)
    return graph.subgraph(largest_nodes).copy()


def sampled_path_statistics(graph, source_samples, seed):
    '''Approximate average shortest path and diameter from sampled BFS sources.'''
    if graph.number_of_nodes() == 0 or source_samples <= 0:
        return {'approx_average_shortest_path': None, 'approx_diameter': None}
    component = largest_component_subgraph(graph)
    rng = random.Random(seed)
    nodes = list(component.nodes)
    sources = rng.sample(nodes, min(source_samples, len(nodes)))

    distance_sum = 0
    reached_pairs = 0
    max_distance = 0
    for source in sources:
        lengths = nx.single_source_shortest_path_length(component, source)
        for target, distance in lengths.items():
            if target == source:
                continue
            distance_sum += distance
            reached_pairs += 1
            max_distance = max(max_distance, distance)
    if reached_pairs == 0:
        return {'approx_average_shortest_path': None, 'approx_diameter': None}
    return {'approx_average_shortest_path': distance_sum / reached_pairs,
            'approx_diameter': max_distance}


def average_clustering_value(graph, trials, seed):
    '''Exact average clustering when affordable, sampled estimate otherwise.'''
    cost_proxy = sum(degree * degree for _, degree in graph.degree())
    if cost_proxy <= CLUSTERING_EXACT_BUDGET:
        return nx.average_clustering(graph), 'exact'
    sampled = nx.algorithms.approximation.average_clustering(
        graph, trials=min(trials, graph.number_of_nodes()), seed=seed)
    return sampled, f'sampled_{trials}'


def graph_property_row(name, graph, seed=SEED):
    '''One row of the model-comparison table, sizes first so mismatches are visible.'''
    n = graph.number_of_nodes()
    m = graph.number_of_edges()
    degrees = [degree for _, degree in graph.degree()]
    component_sizes = sorted((len(c) for c in nx.connected_components(graph)), reverse=True)
    path_stats = sampled_path_statistics(graph, PATH_SOURCE_SAMPLES, seed)
    clustering, clustering_method = average_clustering_value(graph, CLUSTERING_TRIALS, seed)
    return {
        'model': name,
        'nodes': n,
        'edges': m,
        'node_ratio_vs_real': n / N_REAL,
        'edge_ratio_vs_real': m / M_REAL,
        'average_degree': 2 * m / n if n else 0.0,
        'max_degree': max(degrees) if degrees else 0,
        'degree_std': float(np.std(degrees)) if degrees else 0.0,
        'density': nx.density(graph) if n > 1 else 0.0,
        'connected_components': len(component_sizes),
        'largest_component_share': component_sizes[0] / n if n else 0.0,
        'average_clustering': clustering,
        'clustering_method': clustering_method,
        'transitivity': nx.transitivity(graph),
        'approx_average_shortest_path': path_stats['approx_average_shortest_path'],
        'approx_diameter': path_stats['approx_diameter'],
    }


print('Measurement helpers ready.')

## 7. מודלי האפס - ותיקון התאמת הגודל

ארבעה מודלים, שכל אחד מהם עונה על שאלת "מה היה קורה אילו" אחרת:

| מודל | מנגנון | על מה הוא מבקר |
|---|---|---|
| Erdos-Renyi `G(n, m)` | קשתות ממוקמות באופן אחיד אקראי | גודל וצפיפות בלבד |
| configuration model | rewiring אקראי של **רצף הדרגות האמיתי** | גודל, צפיפות *וגם* התפלגות הדרגות |
| Barabasi-Albert | צמיחה + preferential attachment | האם כלל של "העשיר מתעשר" יכול לשחזר את ה-hubs? |
| Watts-Strogatz | סריג טבעתי + rewiring אקראי | האם סריג + קיצורי דרך יכולים לשחזר clustering *וגם* מסלולים קצרים? |

**הבאג והתיקון.** בגרף האמיתי `<k> = 2m/n` שווה בקירוב 3.4, כלומר `m/n` שווה בקירוב 1.7 - ערך שאינו שלם. שני הגנרטורים הקלאסיים מקבלים פרמטרים שלמים בלבד:

- Barabasi-Albert מוסיף `m_BA` קשתות לכל צומת חדש, ולכן הוא יכול לייצר רק כ-`m_BA * n` קשתות. `round(1.7) = 2` נותן כ-61k קשתות - 18% יותר מדי. התיקון שלהלן משתמש ב**פרמטר חיבור שברי**: כל צומת חדש מתחבר ב-1 או ב-2 קשתות, כאשר הבחירה נעשית כך שמספר הקשתות המצטבר עוקב אחר היעד, בעוד הסתברות החיבור נותרת פרופורציונית לדרגה. מנגנון ה-scale-free נותר ללא שינוי; רק תקציב הקשתות נאכף.
- `nx.watts_strogatz_graph` דורש `k` זוגי (כל צומת מקושר ל-`k/2` שכנים בכל צד). הערך `<k> = 3.4` עוגל כלפי מטה לערך הזוגי 2, מה שהוליד טבעת חשופה בעלת כ-30.5k קשתות - 41% פחות מדי, ולטבעת יש clustering השווה בדיוק ל-0 לפני ה-rewiring, מה שמאיין את כל תכליתו של המודל. התיקון בונה סריג טבעתי, ולאחר מכן מוסיף מיתרים (chords) במרחק הולך וגדל לתת-קבוצה אקראית של צמתים עד שתקציב הקשתות מתמלא במדויק, ורק אז מבצע rewiring לכל קשת בהסתברות `p` (rewiring מחליף קצה אחד של הקשת, ולכן מספר הקשתות לעולם אינו משתנה).
- ה-configuration model נשמר כפי שהיה בסקריפט המקורי: הגרלת מולטי-גרף על רצף הדרגות האמיתי, ולאחר מכן קיצוץ קשתות מקבילות והשמטת לולאות עצמיות. פעולה זו מאבדת *בהכרח* מספר קטן של קשתות ואינה ניתנת להתאמת גודל מבלי לשבור את רצף הדרגות - ולכן אנו מדווחים את המספר שהושג בפועל במקום להסתירו.

מדוע הדבר חשוב: הן ה-clustering והן אורך המסלול הממוצע הם מונוטוניים בצפיפות הקשתות. מודל בעל 18% יותר קשתות ייראה כבעל מסלולים קצרים יותר מסיבות שאין להן דבר וחצי דבר עם המנגנון שלו.

In [ ]:
def simple_configuration_graph(degree_sequence, seed):
    '''Configuration model, then collapse parallel edges and drop self-loops.

    Keeps the real degree sequence exactly up to the collapse; the resulting
    (slightly smaller) edge count is reported in the comparison table.
    '''
    multigraph = nx.configuration_model(degree_sequence, seed=seed)
    graph = nx.Graph(multigraph)
    graph.remove_edges_from(nx.selfloop_edges(graph))
    return graph


def _random_subset(repeated_nodes, k, rng):
    '''Pick k distinct nodes with probability proportional to degree.

    `repeated_nodes` contains each node once per incident edge, so a uniform
    draw from it is a degree-proportional draw - the standard preferential
    attachment trick used by networkx itself.
    '''
    targets = set()
    while len(targets) < k:
        targets.add(rng.choice(repeated_nodes))
    return targets


def size_matched_barabasi_albert(n, m_target, seed):
    '''Preferential attachment with a fractional attachment parameter.

    Textbook BA adds an integer number of edges per new node and therefore
    cannot hit a target of m/n = 1.7 edges per node. Here the number of edges
    added by each new node is chosen from the remaining edge budget, so the
    final graph has exactly m_target edges while attachment remains
    degree-proportional.
    '''
    rng = random.Random(seed)
    m_start = max(1, int(math.ceil(m_target / n)))
    graph = nx.empty_graph(m_start)
    repeated_nodes = list(range(m_start))
    for new_node in range(m_start, n):
        remaining_nodes = n - new_node
        needed = m_target - graph.number_of_edges()
        k = int(round(needed / remaining_nodes)) if remaining_nodes else 1
        k = max(1, min(k, new_node))
        targets = _random_subset(repeated_nodes, k, rng)
        graph.add_node(new_node)
        for target in targets:
            graph.add_edge(new_node, target)
        repeated_nodes.extend(targets)
        repeated_nodes.extend([new_node] * k)
    return graph


def size_matched_watts_strogatz(n, m_target, rewire_p, seed):
    '''Watts-Strogatz style small world with an exact edge budget.

    Step 1: ring lattice of nearest neighbours (n edges).
    Step 2: add chords at distance 2, 3, ... to a random subset of nodes until
            the edge budget is met exactly (this is what allows a non-even
            effective k, i.e. an average degree of 3.4 rather than 2 or 4).
    Step 3: rewire each edge with probability rewire_p by moving one endpoint
            to a random node - this creates the shortcuts that give the small
            world its short paths and never changes the edge count.
    '''
    rng = random.Random(seed)
    graph = nx.Graph()
    graph.add_nodes_from(range(n))
    for i in range(n):
        graph.add_edge(i, (i + 1) % n)

    distance = 2
    while graph.number_of_edges() < m_target and distance < n // 2:
        order = list(range(n))
        rng.shuffle(order)
        for i in order:
            if graph.number_of_edges() >= m_target:
                break
            j = (i + distance) % n
            if i != j and not graph.has_edge(i, j):
                graph.add_edge(i, j)
        distance += 1

    if graph.number_of_edges() > m_target:
        surplus = graph.number_of_edges() - m_target
        edges = list(graph.edges())
        rng.shuffle(edges)
        graph.remove_edges_from(edges[:surplus])

    for u, v in list(graph.edges()):
        if rng.random() >= rewire_p:
            continue
        for _ in range(32):
            w = rng.randrange(n)
            if w != u and not graph.has_edge(u, w):
                graph.remove_edge(u, v)
                graph.add_edge(u, w)
                break
    return graph


print('Model generators ready.')

## 8. יצירת המודלים ואימות התאמת הגודל

כעת אנו בונים את כל ארבעת הגרפים הסינתטיים על אותו מספר צמתים כשל הגרף האמיתי, ומדפיסים את מספרי הקשתות שהושגו בפועל זה לצד זה עם המספרים שבחירת הפרמטרים **הישנה והשגויה** הייתה מייצרת. כך התיקון ניתן לביקורת ואינו נותר טענה בפרוזה בלבד. היצירה נמשכת הרבה פחות מדקה; ה-configuration model הוא השלב האיטי ביותר.

In [ ]:
degree_sequence = [degree for _, degree in G.degree()]

models = {}
models['real_transit'] = G
models['erdos_renyi_gnm'] = nx.gnm_random_graph(N_REAL, M_REAL, seed=SEED)
models['configuration_degree_sequence'] = simple_configuration_graph(degree_sequence, SEED)
models['barabasi_albert_matched'] = size_matched_barabasi_albert(N_REAL, M_REAL, SEED)
models['watts_strogatz_matched'] = size_matched_watts_strogatz(N_REAL, M_REAL, WS_REWIRE_P, SEED)

print('Achieved sizes')
for name, graph in models.items():
    n_model = graph.number_of_nodes()
    m_model = graph.number_of_edges()
    print(f'  {name:32s} n={n_model:7,d}  m={m_model:7,d}  edges vs real = {m_model / M_REAL:6.1%}')

# What the previous (buggy) parameter choice would have produced, for contrast.
OLD_BA_M = max(1, int(round(M_REAL / N_REAL)))
OLD_BA_EDGES = OLD_BA_M * (N_REAL - OLD_BA_M)
old_ws_k = max(2, int(round(2 * M_REAL / N_REAL)))
OLD_WS_K = old_ws_k if old_ws_k % 2 == 0 else old_ws_k - 1
OLD_WS_EDGES = N_REAL * OLD_WS_K // 2
print()
print('Previous parameterisation (the bug this notebook fixes)')
print(f'  barabasi_albert m=round(m/n)={OLD_BA_M} would give about {OLD_BA_EDGES:,} edges '
      f'({OLD_BA_EDGES / M_REAL:.1%} of the real network)')
print(f'  watts_strogatz forced-even k={OLD_WS_K} would give exactly {OLD_WS_EDGES:,} edges '
      f'({OLD_WS_EDGES / M_REAL:.1%} of the real network)')

## 9. טבלת ההשוואה

כל גרף מוזרם כעת דרך אותו צינור מדידה בדיוק. הטבלה מדווחת, לפי הסדר: את הגודל שהושג (`nodes`, `edges`, והיחסים מול הגרף האמיתי), סיכום דרגות, קשירות, clustering (בציון השיטה שבה נעשה שימוש), transitivity, וסטטיסטיקות המסלולים המבוססות על דגימה.

**אזהרת עלות:** זהו התא היקר - `PATH_SOURCE_SAMPLES` סריקות BFS לכל גרף בתוספת חישוב מדויק של clustering ו-transitivity, כ-2-5 דקות בסך הכול. יש להקטין את `PATH_SOURCE_SAMPLES` אם נדרשת הרצה מהירה יותר.

מדריך קריאה: `average_clustering` הוא הממוצע של מקדם ה-clustering המקומי לכל צומת (הוא מעניק משקל רב לצמתים בעלי דרגה נמוכה); `transitivity` הוא יחס המשולשים הגלובלי (הוא מעניק משקל רב ל-hubs). הדיווח על שניהם נעשה במכוון - הם חלוקים זה על זה באופן חד בגרפים בעלי זנב כבד.

In [ ]:
comparison = pd.DataFrame([graph_property_row(name, graph) for name, graph in models.items()])
comparison.to_csv(TABLES / 'network_model_comparison.csv', index=False, encoding='utf-8-sig')
print('Written:', TABLES / 'network_model_comparison.csv')
comparison.round(4)

## 10. התפלגויות הדרגות

טבלת ההשוואה מספקת מספר clustering יחיד לכל גרף; התפלגות הדרגות מציגה את הצורה כולה. עבור כל גרף אנו מטבלים את P(k) ואת ה-CDF המשלים P(K >= k), ולאחר מכן מסכמים באמצעות:

- **ממוצע / סטיית תקן / דרגה מקסימלית** - עד כמה הגרף הטרוגני. בגרף פואסוני סטיית התקן שווה בקירוב לשורש הממוצע; בגרף scale-free סטיית התקן גדולה בהרבה והמקסימום גדול לאין ערוך.
- **hill_alpha_kmin4** - מעריך חוק חזקה שנאמד בשיטת נראות מקסימלית עבור הזנב `k >= 4`. זהו מדד *תיאורי בלבד*: `k_min` נקבע מראש ולא נבחר באופן אופטימלי, ולא בוצע מבחן טיב התאמה, ולכן יש לקרוא אותו כ"עד כמה הזנב כבד" ולא כראיה לכך שההתפלגות היא חוק חזקה.
- **ks_vs_real** - מרחק קולמוגורוב-סמירנוב (הפער המקסימלי בין ה-CDF האמפיריים של הדרגות) בין כל מודל לבין הגרף האמיתי. ערך קטן יותר מציין התאמה טובה יותר; ה-configuration model אמור להיות קרוב לאפס מעצם בנייתו, מה שמשמש גם כבדיקת שפיות לצינור העיבוד.

In [ ]:
def degree_distribution_frame(name, graph):
    '''Long-form degree distribution: P(k), CDF and CCDF for one graph.'''
    counts = Counter(degree for _, degree in graph.degree())
    n = graph.number_of_nodes()
    rows = []
    cumulative = 0
    for degree in sorted(counts):
        nodes = counts[degree]
        cumulative += nodes
        rows.append({'model': name, 'degree': degree, 'nodes': nodes,
                     'probability': nodes / n if n else 0.0,
                     'cumulative_probability': cumulative / n if n else 0.0,
                     'ccdf': 1.0 - (cumulative - nodes) / n if n else 0.0})
    return pd.DataFrame(rows)


def degree_cdf(degrees, max_k):
    counts = np.bincount(np.asarray(degrees, dtype=int), minlength=max_k + 1)
    return np.cumsum(counts) / counts.sum()


def ks_statistic(degrees_a, degrees_b):
    '''Kolmogorov-Smirnov distance between two empirical degree distributions.'''
    max_k = max(max(degrees_a), max(degrees_b))
    return float(np.max(np.abs(degree_cdf(degrees_a, max_k) - degree_cdf(degrees_b, max_k))))


def hill_alpha(degrees, k_min):
    '''MLE power-law exponent for the tail k >= k_min. Descriptive only.'''
    tail = [d for d in degrees if d >= k_min]
    if len(tail) < 20:
        return None
    total = sum(math.log(d / (k_min - 0.5)) for d in tail)
    if total <= 0:
        return None
    return 1.0 + len(tail) / total


distributions = pd.concat(
    [degree_distribution_frame(name, graph) for name, graph in models.items()],
    ignore_index=True)
distributions.to_csv(TABLES / 'degree_distributions.csv', index=False, encoding='utf-8-sig')

real_degrees = [degree for _, degree in G.degree()]
summary_rows = []
for name, graph in models.items():
    degrees = [degree for _, degree in graph.degree()]
    array = np.asarray(degrees)
    summary_rows.append({
        'model': name,
        'nodes': graph.number_of_nodes(),
        'edges': graph.number_of_edges(),
        'mean_degree': float(array.mean()),
        'std_degree': float(array.std()),
        'max_degree': int(array.max()),
        'share_degree_ge_10': float((array >= 10).mean()),
        'isolated_nodes': int((array == 0).sum()),
        'hill_alpha_kmin4': hill_alpha(degrees, K_MIN_TAIL),
        'ks_vs_real': 0.0 if name == 'real_transit' else ks_statistic(degrees, real_degrees),
    })

degree_summary = pd.DataFrame(summary_rows)
degree_summary.to_csv(TABLES / 'degree_distribution_summary.csv', index=False, encoding='utf-8-sig')
print('Written:', TABLES / 'degree_distributions.csv')
print('Written:', TABLES / 'degree_distribution_summary.csv')
degree_summary.round(4)

## 11. איור - התפלגויות דרגות על צירים לוגריתמיים כפולים

שתי פאנלים על צירים לוגריתמיים כפולים. הפאנל השמאלי מציג את P(k) הגולמי, הרועש בזנב משום שכל דרגה גבוהה מיוצגת על ידי קומץ תחנות בלבד. הפאנל הימני מציג את ה-CCDF כלומר P(K >= k), שהיא הדרך התקנית לקרוא זנב כבד: קו ישר שם מעיד על התנהגות דמוית חוק חזקה, בעוד כיפוף חד כלפי מטה מעיד על ניתוק אקספוננציאלי. השוואת העקומה האמיתית מול Erdos-Renyi (פואסוני, צונח כמצוק), Barabasi-Albert (הקו הישר ביותר) ו-Watts-Strogatz (כמעט פונקציית דלתא סביב הדרגה הממוצעת) היא הקריאה הוויזואלית המהירה ביותר של מידת ההטרוגניות של הרשת האמיתית.

In [ ]:
PALETTE = {
    'real_transit': '#111827',
    'erdos_renyi_gnm': '#2563eb',
    'configuration_degree_sequence': '#0f766e',
    'barabasi_albert_matched': '#dc2626',
    'watts_strogatz_matched': '#7c3aed',
}

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for name in models:
    subset = distributions[(distributions['model'] == name) & (distributions['degree'] > 0)]
    colour = PALETTE.get(name, '#6b7280')
    axes[0].scatter(subset['degree'], subset['probability'], s=16, alpha=0.7,
                    color=colour, label=name)
    axes[1].plot(subset['degree'], subset['ccdf'].clip(lower=1e-6),
                 linewidth=1.8, color=colour, label=name)

axes[0].set_title('Degree distribution P(k), log-log')
axes[0].set_ylabel('P(k)')
axes[1].set_title('Complementary CDF P(K >= k), log-log')
axes[1].set_ylabel('P(K >= k)')
for ax in axes:
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel('Degree k')
    ax.grid(True, which='both', alpha=0.2)
axes[1].legend(fontsize=8, loc='lower left')
fig.suptitle('Real transit network vs. random network models: degree distribution')
fig.tight_layout()
fig.savefig(FIGURES / 'degree_distribution_comparison.png', dpi=180)
plt.show()
print('Written:', FIGURES / 'degree_distribution_comparison.png')

## 12. איור - clustering ואורך מסלול זה לצד זה

שלושה פאנלים, אחד לכל מדד מרכזי, משום שהם חיים בסקאלות שונות לחלוטין (clustering נמצא בתחום [0, 1], בעוד אורך המסלול נמדד בעשרות קפיצות) ולא היו ניתנים לקריאה על צירים משותפים. מאחר שכל חמשת הגרפים נושאים כעת (כמעט) את אותו מספר צמתים וקשתות, ההבדלים כאן ניתנים לייחוס למבנה ולא לצפיפות.

In [ ]:
metric_specs = [
    ('average_clustering', 'Average clustering coefficient'),
    ('transitivity', 'Transitivity (global triangle ratio)'),
    ('approx_average_shortest_path', 'Approx. average shortest path (hops)'),
]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
labels = comparison['model'].tolist()
colours = [PALETTE.get(name, '#6b7280') for name in labels]
for ax, (column, title) in zip(axes, metric_specs):
    values = pd.to_numeric(comparison[column], errors='coerce').fillna(0.0)
    ax.bar(range(len(labels)), values, color=colours)
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=25, ha='right', fontsize=8)
    ax.set_title(title, fontsize=11)
    ax.grid(True, axis='y', alpha=0.2)
    for x, value in enumerate(values):
        ax.annotate(f'{value:.3g}', (x, value), ha='center', va='bottom', fontsize=8)

fig.suptitle('Structural metrics at matched size (same nodes, same edge budget)')
fig.tight_layout()
fig.savefig(FIGURES / 'model_metric_comparison.png', dpi=180)
plt.show()
print('Written:', FIGURES / 'model_metric_comparison.png')

## 13. איור - ביקורת התאמת הגודל

איור זה קיים אך ורק כדי להפוך את תיקון הבאג לגלוי לעין. הוא משרטט את מספר הקשתות שהושג בפועל בכל מודל מול הגרף האמיתי (קו מקווקו), יחד עם שני העמודים האפורים המראים מה הפרמטריזציה הקודמת הייתה מייצרת. כל עמוד שאינו יושב על הקו המקווקו מייצג אי-התאמה שנותרה, שהקורא זכאי לדעת עליה - ובפועל מדובר רק ב-configuration model, המאבד קשתות בעת קיצוץ קשתות מקבילות.

In [ ]:
audit_labels = list(models.keys()) + ['barabasi_albert (old, m=2)', 'watts_strogatz (old, even k)']
audit_values = [graph.number_of_edges() for graph in models.values()] + [OLD_BA_EDGES, OLD_WS_EDGES]
audit_colours = [PALETTE.get(name, '#6b7280') for name in models] + ['#9ca3af', '#9ca3af']

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(range(len(audit_labels)), audit_values, color=audit_colours)
ax.axhline(M_REAL, color='#111827', linestyle='--', linewidth=1.4,
           label=f'real edge count = {M_REAL:,}')
ax.set_xticks(range(len(audit_labels)))
ax.set_xticklabels(audit_labels, rotation=25, ha='right', fontsize=8)
ax.set_ylabel('Undirected edges')
ax.set_title('Size-match audit: achieved edge counts vs. the real network')
ax.grid(True, axis='y', alpha=0.2)
for x, value in enumerate(audit_values):
    ax.annotate(f'{value / M_REAL:.0%}', (x, value), ha='center', va='bottom', fontsize=8)
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES / 'size_match_check.png', dpi=180)
plt.show()
print('Written:', FIGURES / 'size_match_check.png')

## 14. אבחון small-world

מבחן ה-small-world הקלאסי משווה גרף לגרף אקראי בעל אותו גודל: ב-small world מתקיים `C / C_random >> 1` (מקובץ מקומית) תוך שמירה על `L / L_random` קרוב ל-1 (קצר גלובלית). מדד ה-small-world הוא `sigma = (C / C_rand) / (L / L_rand)`, כאשר `sigma > 1` נחשב מקובל כראיה ל-small-world. כאן גרף ה-Erdos-Renyi ממלא את תפקיד ההתייחסות האקראית, וזו בדיוק המטרה שלשמה נוצר. הסתייגות: `L` הוא אומדן מבוסס דגימה, ו-`C_random` הוא מספר קטן מאוד, ולכן `sigma` רגיש לרעש במכנה - יש להתייחס אליו כאל סדר גודל ולא כאל סטטיסטי מדויק.

In [ ]:
def safe_ratio(numerator, denominator):
    if numerator is None or denominator is None:
        return None
    try:
        if float(denominator) == 0.0:
            return None
        return float(numerator) / float(denominator)
    except (TypeError, ValueError):
        return None


indexed = comparison.set_index('model')
reference = indexed.loc['erdos_renyi_gnm']
c_random = reference['average_clustering']
l_random = reference['approx_average_shortest_path']

small_world_rows = []
for name in indexed.index:
    row = indexed.loc[name]
    c_ratio = safe_ratio(row['average_clustering'], c_random)
    l_ratio = safe_ratio(row['approx_average_shortest_path'], l_random)
    small_world_rows.append({
        'model': name,
        'clustering_C': row['average_clustering'],
        'avg_path_L': row['approx_average_shortest_path'],
        'C_over_C_random': c_ratio,
        'L_over_L_random': l_ratio,
        'small_world_sigma': safe_ratio(c_ratio, l_ratio),
    })

small_world = pd.DataFrame(small_world_rows)
small_world.to_csv(TABLES / 'small_world_ratios.csv', index=False, encoding='utf-8-sig')
print('Written:', TABLES / 'small_world_ratios.csv')
small_world.round(3)

## 15. פסק דין מונחה-נתונים

במקום לקבוע מסקנות בפרוזה שעלולות להתנתק מהמציאות בהרצה חוזרת, תא זה מדפיס פסק דין המורכב מהמספרים שחושבו בפועל למעלה: איזה מודל התאים בגודל, איזה התאים בהתפלגות הדרגות (מרחק KS הנמוך ביותר), עד כמה רחוק ה-clustering האמיתי מקו הבסיס האקראי, והאם אורך המסלול הממוצע האמיתי קרוב בכלל לזה של הגרף האקראי.

In [ ]:
def fmt(value, digits=4):
    if value is None:
        return 'n/a'
    try:
        number = float(value)
    except (TypeError, ValueError):
        return str(value)
    if math.isnan(number):
        return 'n/a'
    return f'{number:.{digits}f}'


real_row = indexed.loc['real_transit']
ks_ranking = degree_summary[degree_summary['model'] != 'real_transit'].sort_values('ks_vs_real')
best_degree_model = ks_ranking.iloc[0]
sw_indexed = small_world.set_index('model')
real_sw = sw_indexed.loc['real_transit']

print('=== Size match ===')
for name in indexed.index:
    ratio = indexed.loc[name, 'edge_ratio_vs_real']
    node_ratio = indexed.loc[name, 'node_ratio_vs_real']
    print(f'  {name:32s} nodes {node_ratio:6.1%}   edges {ratio:6.1%}')

print()
print('=== Degree distribution ===')
for _, row in degree_summary.iterrows():
    print(f'  {row["model"]:32s} mean {row["mean_degree"]:5.2f}  std {row["std_degree"]:5.2f}  '
          f'max {int(row["max_degree"]):4d}  KS vs real {row["ks_vs_real"]:.4f}')
print(f'  Closest degree distribution to the real graph: {best_degree_model["model"]} '
      f'(KS = {best_degree_model["ks_vs_real"]:.4f})')

print()
print('=== Clustering and path length ===')
print(f'  real   C = {fmt(real_row["average_clustering"])}   L = {fmt(real_row["approx_average_shortest_path"], 2)}')
print(f'  random C = {fmt(c_random)}   L = {fmt(l_random, 2)}')
print(f'  real C / random C = {fmt(real_sw["C_over_C_random"], 1)}   '
      f'real L / random L = {fmt(real_sw["L_over_L_random"], 2)}   '
      f'sigma = {fmt(real_sw["small_world_sigma"], 1)}')

l_ratio_value = real_sw['L_over_L_random']
if l_ratio_value is not None and float(l_ratio_value) > 2:
    print('  -> The real network is far MORE clustered than random but its paths are also much '
          'longer than random: it is not a small world in the Watts-Strogatz sense.')
else:
    print('  -> The real network combines above-random clustering with near-random path lengths: '
          'small-world behaviour.')

print()
print('=== Stage outputs ===')
for path in sorted(TABLES.glob('*.csv')) + sorted(FIGURES.glob('*.png')):
    print(' ', path.relative_to(REPO))

## מסקנות

יש לקרוא אותן מול המספרים שהודפסו בתא פסק הדין - אלו הן המסקנות המבניות שההשוואה תומכת בהן.

1. **תיקון התאמת הגודל משנה את המסקנה, ולא רק את המראה החיצוני.** הפרמטריזציה הישנה השוותה את הרשת האמיתית מול גרף Barabasi-Albert בעל כ-18% יותר מדי קשתות ומול גרף Watts-Strogatz בעל כ-41% פחות מדי. הן ה-clustering והן אורך המסלול משתנים עם הצפיפות, ולכן פערים אלו לבדם יכלו לייצר הבדלים בסדר גודל של אלו שאותם מפרשים. עם גנרטור ה-BA בעל `m` שברי ועם גנרטור ה-Watts-Strogatz מבוסס המיתרים, כל המודלים למעט ה-configuration model נושאים כעת במדויק את מספר הצמתים והקשתות האמיתי; ה-configuration model מאבד בהכרח חלק קטן מהקשתות בעת קיצוץ קשתות מקבילות, ואובדן זה מדווח ב-`edge_ratio_vs_real` במקום להיות מוסתר.

2. **אף מודל קלאסי בודד אינו משחזר את רשת התחבורה.** כל אחד מהם לוכד תכונה אחת ומחמיץ את השאר: Erdos-Renyi מתאים בצפיפות אך ה-clustering שלו אפסי למעשה וריכוז הדרגות שלו סביב הממוצע גבוה בהרבה; ה-configuration model מתאים להתפלגות הדרגות מעצם בנייתו (מרחק ה-KS שלו קרוב לאפס) ואף על פי כן הורס את ה-clustering, ובכך מראה שהמשולשים האמיתיים *אינם* תוצר לוואי של רצף הדרגות; Barabasi-Albert מייצר זנב דרגות כבד אך מעט מדי משולשים ומסלולים קצרים באופן לא ריאלי; Watts-Strogatz מייצר clustering בשפע אך התפלגות דרגות כמעט אחידה שאין לה מקבילה באף מערכת תחבורה אמיתית.

3. **האילוץ הדומיננטי הוא הגיאוגרפיה, ואף אחד מארבעת המודלים אינו מודע לה.** גרף תחנות הוא קרוב למישורי: תחנות מתחברות לשכנים פיזיים לאורך כבישים, ולכן הדרגות אינן יכולות לגדול ללא גבול והמרחקים גדלים בקירוב עם המרחק הגיאוגרפי. זו הסיבה שהמסלול הקצר ביותר הממוצע האמיתי ארוך בהרבה מאשר בכל מודל אקראי בעל אותו גודל - הרשת משוכנת במרחב, ואינה small world. clustering גבוה בהרבה מקו הבסיס האקראי יחד עם אורכי מסלול גבוהים בהרבה מקו הבסיס האקראי הם החתימה של גרף דמוי-סריג המוגבל מרחבית.

4. **מגבלות מוצהרות.** `L` והקוטר נדגמים מ-`PATH_SOURCE_SAMPLES` מקורות BFS בתוך הרכיב הגדול ביותר, ולכן הם אומדנים (הקוטר בפרט הוא חסם תחתון). `hill_alpha_kmin4` הוא מדד זנב תיאורי בעל נקודת חיתוך קבועה וללא מבחן טיב התאמה - הוא אינו מבסס שהתפלגות הדרגות היא חוק חזקה, ובהינתן דרגה מקסימלית בעשרות הנמוכות, הרשת האמיתית מתוארת טוב יותר כבעלת זנב כבד במידה מתונה מאשר כ-scale-free. לבסוף, `sigma` מחלק בערך ה-clustering הקטן מאוד של הגרף האקראי ויש לקרוא אותו כסדר גודל בלבד.